In [45]:
import numpy as np 
import pandas as pd 
from datasets import load_dataset
import re

In [46]:
SEED = 42
SLICE_SIZE = 5000
IDENTITY_THRESHOLD = 0.3
DATA_DIR = "data"
TAG_PATTERN = re.compile(r"^.{0,30}?H{6,}")

In [47]:
ds = load_dataset("SeprotHub/Dataset-Solubility")
ds

DatasetDict({
    train: Dataset({
        features: ['protein', 'label', 'stage'],
        num_rows: 71419
    })
})

In [48]:
df = ds["train"].to_pandas()
print(df.shape)
df.head()

(71419, 3)


,protein,label,stage
0,GMILKTNLFGHTYQFKSITDVLAKANEEKSGDRLAGVAAESAEERV...,1,train
1,MAHHHHHHMSFFRMKRRLNFVVKRGIEELWENSFLDNNVDMKKIEY...,0,train
2,MGSDKIHHHHHHMEKSIQDTIHGVIKLEDWMVEIVDTPQFQRLRRI...,0,train
3,MEKYIHSVEDYHRLISYLENNLNYEDSVVNHVIYVIAKTGMRYGEI...,0,train
4,MSLTDSFTVRSIEGVCFRYPLATPVVTSFGKMLNRPAVFVRVVDED...,0,train


In [50]:
print(df["stage"].value_counts())
print(df["label"].value_counts())

stage
train    62478
valid     6942
test      1999
Name: count, dtype: int64
label
0    41447
1    29972
Name: count, dtype: int64


In [51]:
pd.crosstab(df["stage"], df["label"], normalize="index")

label,0,1
stage,,
test,0.499750,0.500250
train,0.582653,0.417347
valid,0.582685,0.417315


In [52]:
df['length'] = df['protein'].str.len()
df['length'].describe()

count    71419.000000
mean       298.877778
std        162.423433
min         19.000000
25%        185.000000
50%        275.000000
75%        377.000000
max       1697.000000
Name: length, dtype: float64

In [53]:
df.groupby("label")["length"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,41447.0,305.608826,157.293440,22.0,195.0,286.0,384.0,1697.0
1,29972.0,289.569698,168.822574,19.0,172.0,259.0,366.0,1691.0


In [54]:
all_chars = set("".join(df["protein"]))
all_chars
print(len(all_chars))

20


In [55]:
df["protein"].duplicated().sum()

np.int64(0)

In [65]:
N_PATTERN = r"^.{0,30}?H{6,}"
C_PATTERN = r"H{6,}.{0,10}$"

df["has_n"] = df["protein"].str.contains(N_PATTERN, regex=True)
df["has_c"] = df["protein"].str.contains(C_PATTERN, regex=True)

df["has_n"].sum(), df["has_c"].sum(), (df["has_n"] & df["has_c"]).sum()

(np.int64(17332), np.int64(14850), np.int64(7))

In [66]:
pd.crosstab(df["has_c"], df["label"], normalize="index")

label,0,1
has_c,,
False,0.627216,0.372784
True,0.401751,0.598249


In [67]:
N_PATTERN = r"^.{0,30}?H{6,}"
C_PATTERN = r"H{6,}.{0,10}$"
MIN_LEN = 20

# What each sequence would look like after removal
n_removed = df["protein"].str.replace(N_PATTERN, "", n=1, regex=True)
c_removed = df["protein"].str.replace(C_PATTERN, "", n=1, regex=True)

# Only count it as a tag if a real protein survives
df["tag_n"] = df["protein"].str.contains(N_PATTERN, regex=True) & (n_removed.str.len() >= MIN_LEN)
df["tag_c"] = df["protein"].str.contains(C_PATTERN, regex=True) & (c_removed.str.len() >= MIN_LEN)

# Strip, N-terminal first, then C-terminal on the result
df["protein_stripped"] = df["protein"]
df.loc[df["tag_n"], "protein_stripped"] = (
    df.loc[df["tag_n"], "protein_stripped"].str.replace(N_PATTERN, "", n=1, regex=True)
)
df.loc[df["tag_c"], "protein_stripped"] = (
    df.loc[df["tag_c"], "protein_stripped"].str.replace(C_PATTERN, "", n=1, regex=True)
)

In [68]:
print(df.shape)
print(df["protein_stripped"].str.len().min())
print(df["protein_stripped"].str.match(r"^H{6,}").sum())
print(df["protein_stripped"].str.contains(r"H{6,}$", regex=True).sum())
print(df["tag_n"].sum(), df["tag_c"].sum())

(71419, 12)
19
0
0
17330 14850


In [69]:
df["length_original"] = df["protein"].str.len()
df["length"] = df["protein_stripped"].str.len()
df.to_parquet("data/solubility_clean.parquet")